# FLEX-Med Centralized Training Benchmark

**Purpose**: Train ResNet50, MobileNetV2, and DenseNet121 independently on Dirichlet-partitioned data to establish centralized baselines for comparison with federated learning (FedMD) results.

**Key**: Uses the **exact same** Dirichlet partitioning, transforms, loss functions, and training pipeline as the FL simulation — minus knowledge distillation and consensus aggregation.

In [ ]:
# Cell 1: Setup & Imports
from google.colab import drive
drive.mount('/content/drive')

!pip install -q flwr-datasets datasets torch torchvision scikit-learn seaborn matplotlib pandas

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.transforms import Compose, ToTensor, Normalize
from sklearn.metrics import roc_auc_score, confusion_matrix
from flwr_datasets.partitioner import DirichletPartitioner
from datasets import Dataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Cell 2: Configuration
# ============================================================================
# CONFIGURE: Just change HETEROGENEITY_PRESET below.
# All preset-dependent parameters auto-configure from db.sql definitions.
# ============================================================================

# --- Heterogeneity Presets (mirrored from db.sql system_config) ---
PRESETS = {
    "low": {
        "dirichlet_alpha": 2.0,
        "minority_boost": 0.70,
        "focal_alpha": 0.50,
        "focal_gamma": 2.0,
        "lr_decay": 0.92,
        "learning_rate": 0.001,
    },
    "moderate": {
        "dirichlet_alpha": 1.0,
        "minority_boost": 0.80,
        "focal_alpha": 0.50,
        "focal_gamma": 2.0,
        "lr_decay": 0.88,
        "learning_rate": 0.001,
    },
    "high": {
        "dirichlet_alpha": 0.5,
        "minority_boost": 0.85,
        "focal_alpha": 0.50,
        "focal_gamma": 2.5,
        "lr_decay": 0.85,
        "learning_rate": 0.001,
    },
}

# ============================================================================
# >>> CHANGE THIS to switch between runs <<<
HETEROGENEITY_PRESET = "high"  # "low" (alpha=2.0), "moderate" (alpha=1.0), "high" (alpha=0.5)
# ============================================================================

assert HETEROGENEITY_PRESET in PRESETS, f"Unknown preset: {HETEROGENEITY_PRESET}. Choose from {list(PRESETS.keys())}"
preset = PRESETS[HETEROGENEITY_PRESET]

# --- Auto-configured from preset ---
ALPHA = preset["dirichlet_alpha"]
INITIAL_LR = preset["learning_rate"]
LR_DECAY = preset["lr_decay"]
FOCAL_ALPHA = preset["focal_alpha"]
FOCAL_GAMMA = preset["focal_gamma"]
MINORITY_BOOST = preset["minority_boost"]

# --- Fixed parameters (shared across all presets, from db.sql) ---
NUM_PARTITIONS = 3             # Must match FL simulation num-supernodes
SEED = 42                      # Must match FL simulation seed
MIN_PARTITION_SIZE = 400       # Must match FL simulation
TOTAL_EPOCHS = 20              # 10 FL rounds x 2 local-epochs = 20 total
EPOCHS_PER_ROUND = 2           # Local epochs per FL round
BATCH_SIZE = 32
WEIGHT_DECAY = 0.01            # AdamW weight decay
GRAD_CLIP_NORM = 1.0           # Gradient clipping max norm
LABEL_SMOOTHING = 0.05         # Label smoothing
NUM_CLASSES = 2
IMG_SIZE = 224

# --- Partition-to-Model Assignment ---
# Must match your FL simulation's client-partition mapping.
MODEL_PARTITION_MAP = {
    "resnet50": 0,             # Partition 0 -> ResNet50
    "mobilenet_v2": 1,         # Partition 1 -> MobileNetV2
    "densenet121": 2,          # Partition 2 -> DenseNet121
}

# --- Dataset Paths (Google Drive) ---
TRAIN_DATASET_PATH = "/content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc/cnmc_local_train"
TEST_DATASET_PATH = "/content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc/cnmc_public_test"

# --- Model Checkpoint Save Path ---
CHECKPOINT_DIR = f"/content/drive/MyDrive/College/FLEX-Med/benchmarks/centralized_alpha_{ALPHA}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Preset: {HETEROGENEITY_PRESET.upper()}")
print(f"  Dirichlet alpha:  {ALPHA}")
print(f"  LR decay:         {LR_DECAY}")
print(f"  Focal gamma:      {FOCAL_GAMMA}")
print(f"  Minority boost:   {MINORITY_BOOST}")
print(f"  Focal alpha:      {FOCAL_ALPHA}")
print(f"  Initial LR:       {INITIAL_LR}")
print(f"  Total epochs:     {TOTAL_EPOCHS} ({TOTAL_EPOCHS // EPOCHS_PER_ROUND} rounds x {EPOCHS_PER_ROUND} epochs/round)")
print(f"  Batch size:       {BATCH_SIZE}")
print(f"  Models:           {list(MODEL_PARTITION_MAP.keys())}")
print(f"  Checkpoints:      {CHECKPOINT_DIR}")

In [ ]:
# Cell 3: Data Loading Functions
# Replicates the exact Dirichlet partitioning from task.py

# --- Transforms (identical to task.py) ---
COMMON_TRANSFORM = Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

PRIVATE_TRAIN_TRANSFORM = Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class TransformOverrideSubset(torch.utils.data.Dataset):
    """Wraps a nested Subset to apply a different transform (e.g. no augmentation for val)."""

    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
        ds = subset
        self._index_chain = []
        while isinstance(ds, Subset):
            self._index_chain.append(ds.indices)
            ds = ds.dataset
        self._root_dataset = ds

    def _resolve_index(self, idx):
        resolved = idx
        for indices in self._index_chain:
            resolved = indices[resolved]
        return resolved

    def __getitem__(self, idx):
        root_idx = self._resolve_index(idx)
        path, label = self._root_dataset.samples[root_idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

    def __len__(self):
        return len(self.subset)


def create_dirichlet_partitioner(dataset_path, num_partitions, alpha, seed, min_partition_size):
    """Create a Dirichlet partitioner (identical to task.py)."""
    full_dataset = datasets.ImageFolder(root=dataset_path, transform=None)
    image_paths = [path for path, _ in full_dataset.samples]
    labels = [label for _, label in full_dataset.samples]

    hf_dataset = Dataset.from_dict({"image_path": image_paths, "label": labels})
    partitioner = DirichletPartitioner(
        num_partitions=num_partitions, partition_by="label", alpha=alpha,
        min_partition_size=min_partition_size, self_balancing=False, shuffle=True, seed=seed
    )
    partitioner.dataset = hf_dataset
    return partitioner, full_dataset


def load_partition_dataset(partition_id, partitioner, full_dataset, batch_size):
    """Load a Dirichlet partition with train/val split (identical to task.py logic)."""
    partition_dataset = partitioner.load_partition(partition_id)
    partition_paths = partition_dataset["image_path"]

    path_to_idx = {path: idx for idx, (path, _) in enumerate(full_dataset.samples)}
    client_indices = [path_to_idx[path] for path in partition_paths]

    client_dataset = Subset(
        datasets.ImageFolder(root=TRAIN_DATASET_PATH, transform=PRIVATE_TRAIN_TRANSFORM),
        client_indices
    )

    # Stratified 85/15 train/val split
    class_indices = {0: [], 1: []}
    for subset_idx, global_idx in enumerate(client_indices):
        class_indices[full_dataset.targets[global_idx]].append(subset_idx)

    train_indices, val_indices = [], []
    generator = torch.Generator().manual_seed(42)

    for indices in class_indices.values():
        if not indices:
            continue
        perm = torch.randperm(len(indices), generator=generator)
        shuffled = torch.tensor(indices)[perm].tolist()
        split = int(0.85 * len(shuffled))
        train_indices.extend(shuffled[:split])
        val_indices.extend(shuffled[split:])

    train_ds = Subset(client_dataset, train_indices)
    val_ds = TransformOverrideSubset(Subset(client_dataset, val_indices), COMMON_TRANSFORM)

    # Weighted random sampler for class imbalance
    train_targets = [full_dataset.targets[client_indices[i]] for i in train_indices]
    class_sample_counts = torch.bincount(torch.tensor(train_targets), minlength=2).float()
    minority_class = torch.argmin(class_sample_counts).item()
    class_weights = 1.0 / class_sample_counts.clamp(min=1)
    if MINORITY_BOOST != 1.0:
        class_weights[minority_class] *= MINORITY_BOOST
    sample_weights = class_weights[torch.tensor(train_targets)]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    trainloader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2)
    valloader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    class_counts = {0: train_targets.count(0), 1: train_targets.count(1)}
    return trainloader, valloader, class_counts


# --- Create partitioner and print stats ---
print("Creating Dirichlet partitioner...")
partitioner, full_dataset = create_dirichlet_partitioner(
    TRAIN_DATASET_PATH, NUM_PARTITIONS, ALPHA, SEED, MIN_PARTITION_SIZE
)

print(f"\n{'='*80}")
print(f"PARTITION STATISTICS (alpha={ALPHA}, seed={SEED}, partitions={NUM_PARTITIONS})")
print(f"{'='*80}")

total_all, total_healthy = 0, 0
for pid in range(NUM_PARTITIONS):
    partition = partitioner.load_partition(pid)
    labels = partition["label"]
    n_all = sum(l == 0 for l in labels)
    n_healthy = sum(l == 1 for l in labels)
    total = len(labels)
    all_pct = n_all / total * 100
    healthy_pct = n_healthy / total * 100
    ratio = n_all / n_healthy if n_healthy > 0 else float('inf')
    total_all += n_all
    total_healthy += n_healthy

    # Find which model uses this partition
    assigned_model = next((m for m, p in MODEL_PARTITION_MAP.items() if p == pid), "unassigned")
    print(f"Partition {pid} ({assigned_model}): Total={total:4d} | "
          f"ALL={n_all:4d} ({all_pct:5.1f}%) | Healthy={n_healthy:4d} ({healthy_pct:5.1f}%) | "
          f"Ratio={ratio:.2f}:1")

overall_total = total_all + total_healthy
overall_ratio = total_all / total_healthy if total_healthy > 0 else float('inf')
print(f"-" * 80)
print(f"OVERALL: Total={overall_total:4d} | Ratio={overall_ratio:.2f}:1")
print(f"{'='*80}")

In [ ]:
# Cell 4: Model & Training Functions

# --- Model Creation (identical to task.py) ---

def get_initial_dropout_rate(model_type):
    """Get default dropout rate for model architecture."""
    rates = {'resnet50': 0.35, 'mobilenet_v2': 0.45, 'densenet121': 0.35}
    return rates.get(model_type.lower(), 0.3)


def add_dropout_to_classifier(model, model_type, dropout_rate=0.3):
    """Add dropout layer before the final classifier head."""
    model_type = model_type.lower()
    if model_type == 'resnet50':
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(in_features, NUM_CLASSES))
    elif model_type == 'mobilenet_v2':
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(in_features, NUM_CLASSES))
    elif model_type == 'densenet121':
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(in_features, NUM_CLASSES))
    return model


def get_model_by_type(model_type):
    """Create a model with pretrained weights and dropout-enhanced classifier."""
    model_type = model_type.lower()
    dropout_rate = get_initial_dropout_rate(model_type)
    model_map = {
        'resnet50': models.resnet50,
        'mobilenet_v2': models.mobilenet_v2,
        'densenet121': models.densenet121,
    }
    model = model_map[model_type](weights="DEFAULT")
    return add_dropout_to_classifier(model, model_type, dropout_rate)


# --- Focal Loss (identical to task.py) ---

class FocalLoss(nn.Module):
    """Focal Loss: FL(pt) = -alpha * (1 - pt)^gamma * log(pt)."""

    def __init__(self, alpha=0.35, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce_loss)
        focal_term = (1 - pt) ** self.gamma
        alpha_t = torch.where(targets == 0, self.alpha, 1 - self.alpha)
        return (alpha_t * focal_term * ce_loss).mean()


# --- Freeze Strategy (adapted from helpers.py for epoch-based training) ---

def freeze_backbone(model, model_type):
    """Freeze all backbone layers, keep only classifier trainable."""
    model_type = model_type.lower()
    for param in model.parameters():
        param.requires_grad = False

    if model_type in ('resnet50', 'resnet18'):
        for param in model.fc.parameters():
            param.requires_grad = True
    elif model_type == 'mobilenet_v2':
        for param in model.classifier.parameters():
            param.requires_grad = True
    elif model_type == 'densenet121':
        for param in model.classifier.parameters():
            param.requires_grad = True
    return model


def unfreeze_last_block(model, model_type):
    """Unfreeze the last backbone block + classifier."""
    model_type = model_type.lower()
    if model_type in ('resnet50', 'resnet18'):
        for param in model.layer4.parameters():
            param.requires_grad = True
    elif model_type == 'mobilenet_v2':
        features_list = list(model.features.children())
        for block in features_list[-3:]:
            for param in block.parameters():
                param.requires_grad = True
    elif model_type == 'densenet121':
        for param in model.features.denseblock4.parameters():
            param.requires_grad = True
        if hasattr(model.features, 'transition3'):
            for param in model.features.transition3.parameters():
                param.requires_grad = True
    return model


def apply_freeze_strategy(model, model_type, epoch, total_epochs):
    """
    Apply freeze strategy based on epoch (maps to FL round phases).
    Phase 1 (first 20% of epochs): Classifier only
    Phase 2 (remaining 80%): Last block + classifier
    """
    phase1_end = max(2 * EPOCHS_PER_ROUND, int(total_epochs * 0.20))  # 4 epochs for 20 total

    if epoch <= phase1_end:
        model = freeze_backbone(model, model_type)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        phase = f"Phase 1 - Classifier only ({trainable:,} params)"
    else:
        model = freeze_backbone(model, model_type)
        model = unfreeze_last_block(model, model_type)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        phase = f"Phase 2 - Last block + classifier ({trainable:,} params)"

    return model, phase


# --- Training & Validation ---

def train_one_epoch(model, trainloader, optimizer, criterion, device):
    """Train for a single epoch. Returns (avg_loss, accuracy)."""
    model.train()
    total_loss, total_batches = 0.0, 0
    correct, total = 0, 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    return total_loss / max(total_batches, 1), correct / max(total, 1)


def validate(model, valloader, criterion, device):
    """Validate model. Returns (avg_loss, accuracy)."""
    model.eval()
    total_loss, total_batches = 0.0, 0
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in valloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            total_batches += 1
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return total_loss / max(total_batches, 1), correct / max(total, 1)


# --- Evaluation (identical metrics to task.py test()) ---

def evaluate(model, testloader, device):
    """Comprehensive evaluation with all metrics matching task.py test()."""
    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss()
    correct, total, total_loss = 0, 0, 0.0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            probs = F.softmax(outputs, dim=1)
            all_probs.extend(probs.cpu().tolist())
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    accuracy = correct / total if total > 0 else 0.0
    loss = total_loss / len(testloader) if len(testloader) > 0 else 0.0
    all_preds = torch.tensor(all_preds)
    all_labels = torch.tensor(all_labels)
    all_probs = torch.tensor(all_probs)

    # Confusion matrix (class 0 = Leukemia/ALL, class 1 = Healthy)
    TP = ((all_preds == 0) & (all_labels == 0)).sum().item()
    FP = ((all_preds == 0) & (all_labels == 1)).sum().item()
    FN = ((all_preds == 1) & (all_labels == 0)).sum().item()
    TN = ((all_preds == 1) & (all_labels == 1)).sum().item()

    leukemia_acc = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    healthy_acc = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = leukemia_acc
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    specificity = healthy_acc
    class_gap = abs(healthy_acc - leukemia_acc)

    try:
        roc_auc = roc_auc_score(all_labels.numpy(), all_probs[:, 0].numpy())
    except Exception:
        roc_auc = (recall + specificity) / 2

    return {
        "loss": round(loss, 6),
        "accuracy": round(accuracy, 6),
        "precision": round(precision, 6),
        "recall": round(recall, 6),
        "f1_score": round(f1_score, 6),
        "specificity": round(specificity, 6),
        "roc_auc": round(roc_auc, 6),
        "leukemia_accuracy": round(leukemia_acc, 6),
        "healthy_accuracy": round(healthy_acc, 6),
        "class_gap": round(class_gap, 6),
        "confusion_matrix": {"TP": TP, "FP": FP, "FN": FN, "TN": TN},
        "num_samples": total,
        "all_preds": all_preds.numpy(),
        "all_labels": all_labels.numpy(),
    }


print("Functions defined successfully.")

In [ ]:
# Cell 5: Training Loop

trained_models = {}
training_history = {}

num_simulated_rounds = TOTAL_EPOCHS // EPOCHS_PER_ROUND

for model_type, partition_id in MODEL_PARTITION_MAP.items():
    print(f"\n{'='*70}")
    print(f"TRAINING: {model_type.upper()} on Partition {partition_id}")
    print(f"{'='*70}")

    # Load data partition
    trainloader, valloader, class_counts = load_partition_dataset(
        partition_id, partitioner, full_dataset, BATCH_SIZE
    )
    n_leukemia = class_counts.get(0, 0)
    n_healthy = class_counts.get(1, 0)
    print(f"Training samples: {n_leukemia + n_healthy} (Leukemia: {n_leukemia}, Healthy: {n_healthy})")
    print(f"Validation samples: {len(valloader.dataset)}")

    # Create model
    model = get_model_by_type(model_type)
    model.to(DEVICE)

    criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": [],
        "lr": [], "phase": []
    }

    start_time = time.time()

    for epoch in range(1, TOTAL_EPOCHS + 1):
        # Determine simulated FL round (1-indexed)
        simulated_round = (epoch - 1) // EPOCHS_PER_ROUND + 1

        # Apply freeze strategy
        model, phase_desc = apply_freeze_strategy(model, model_type, epoch, TOTAL_EPOCHS)

        # Calculate decayed LR (per simulated round, matching FL)
        current_lr = INITIAL_LR * (LR_DECAY ** (simulated_round - 1))

        # Optimizer with only trainable params (respects freeze)
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=current_lr, betas=(0.9, 0.999), weight_decay=WEIGHT_DECAY
        )
        scheduler = CosineAnnealingLR(optimizer, T_max=1)  # Per-epoch cosine

        # Train
        train_loss, train_acc = train_one_epoch(model, trainloader, optimizer, criterion, DEVICE)
        scheduler.step()

        # Validate
        val_loss, val_acc = validate(model, valloader, criterion, DEVICE)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)
        history["phase"].append(phase_desc)

        if epoch % 2 == 0 or epoch == 1:  # Print every simulated round boundary + first epoch
            print(f"  Epoch {epoch:2d}/{TOTAL_EPOCHS} (Round {simulated_round:2d}) | "
                  f"LR: {current_lr:.6f} | "
                  f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2%} | "
                  f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2%}")

    elapsed = time.time() - start_time
    print(f"\nTraining completed in {elapsed:.1f}s")
    print(f"Final - Train Acc: {history['train_acc'][-1]:.2%}, Val Acc: {history['val_acc'][-1]:.2%}")

    # Save model checkpoint
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{model_type}_partition{partition_id}.pt")
    torch.save({
        'model_type': model_type,
        'num_classes': NUM_CLASSES,
        'state_dict': model.state_dict(),
        'partition_id': partition_id,
        'alpha': ALPHA,
        'final_val_acc': history['val_acc'][-1],
    }, checkpoint_path)
    print(f"Model saved: {checkpoint_path}")

    trained_models[model_type] = model
    training_history[model_type] = history

print(f"\n{'='*70}")
print("ALL MODELS TRAINED SUCCESSFULLY")
print(f"{'='*70}")

In [ ]:
# Cell 6: Evaluation on Unseen Test Set

print(f"Loading test dataset from: {TEST_DATASET_PATH}")
test_dataset = datasets.ImageFolder(root=TEST_DATASET_PATH, transform=COMMON_TRANSFORM)
testloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Test samples: {len(test_dataset)} (Classes: {test_dataset.class_to_idx})")

eval_results = {}

for model_type, model in trained_models.items():
    print(f"\nEvaluating {model_type.upper()}...")
    metrics = evaluate(model, testloader, DEVICE)
    eval_results[model_type] = metrics

    cm = metrics["confusion_matrix"]
    print(f"  Accuracy:    {metrics['accuracy']:.4f}")
    print(f"  Precision:   {metrics['precision']:.4f}")
    print(f"  Recall:      {metrics['recall']:.4f}")
    print(f"  F1 Score:    {metrics['f1_score']:.4f}")
    print(f"  Specificity: {metrics['specificity']:.4f}")
    print(f"  ROC-AUC:     {metrics['roc_auc']:.4f}")
    print(f"  Leukemia Acc: {metrics['leukemia_accuracy']:.4f}")
    print(f"  Healthy Acc:  {metrics['healthy_accuracy']:.4f}")
    print(f"  Class Gap:    {metrics['class_gap']:.4f}")
    print(f"  CM: TP={cm['TP']}, FP={cm['FP']}, FN={cm['FN']}, TN={cm['TN']}")

In [ ]:
# Cell 7: Results Comparison Table

metric_names = [
    "accuracy", "precision", "recall", "f1_score",
    "specificity", "roc_auc", "leukemia_accuracy",
    "healthy_accuracy", "class_gap", "loss"
]

table_data = {}
for model_type in MODEL_PARTITION_MAP.keys():
    if model_type in eval_results:
        partition_id = MODEL_PARTITION_MAP[model_type]
        col_name = f"{model_type}\n(Partition {partition_id})"
        table_data[col_name] = [
            f"{eval_results[model_type][m]:.4f}" for m in metric_names
        ]

df = pd.DataFrame(table_data, index=[m.replace('_', ' ').title() for m in metric_names])

print(f"\n{'='*80}")
print(f"CENTRALIZED TRAINING BENCHMARK RESULTS (alpha={ALPHA})")
print(f"{'='*80}")
print(df.to_string())
print(f"{'='*80}")

# Also display as a styled DataFrame in Colab
df

In [ ]:
# Cell 8: Confusion Matrices

model_names = list(eval_results.keys())
n_models = len(model_names)
class_labels = ["ALL (Leukemia)", "Healthy"]

fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1:
    axes = [axes]

for idx, model_type in enumerate(model_names):
    results = eval_results[model_type]
    cm = confusion_matrix(results["all_labels"], results["all_preds"])

    # Calculate percentages
    cm_pct = cm.astype('float') / cm.sum() * 100

    # Create annotation strings with counts and percentages
    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]}\n({cm_pct[i, j]:.1f}%)"

    partition_id = MODEL_PARTITION_MAP[model_type]
    sns.heatmap(
        cm, annot=annot, fmt='', cmap='Blues', ax=axes[idx],
        xticklabels=class_labels, yticklabels=class_labels,
        cbar=True, square=True, linewidths=0.5
    )
    axes[idx].set_title(f"{model_type.upper()}\n(Partition {partition_id})\n"
                        f"Acc: {results['accuracy']:.2%}, F1: {results['f1_score']:.2%}",
                        fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

fig.suptitle(f'Centralized Training - Confusion Matrices (Dirichlet alpha={ALPHA})',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, f"confusion_matrices_alpha_{ALPHA}.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {CHECKPOINT_DIR}/confusion_matrices_alpha_{ALPHA}.png")

In [ ]:
# Cell 9: Training Curves

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
colors = {'resnet50': '#1f77b4', 'mobilenet_v2': '#ff7f0e', 'densenet121': '#2ca02c'}
epochs_range = range(1, TOTAL_EPOCHS + 1)

# --- Loss Plot ---
for model_type in MODEL_PARTITION_MAP.keys():
    if model_type not in training_history:
        continue
    h = training_history[model_type]
    color = colors.get(model_type, 'gray')
    partition_id = MODEL_PARTITION_MAP[model_type]
    label = f"{model_type} (P{partition_id})"
    axes[0].plot(epochs_range, h["train_loss"], '-', color=color, alpha=0.6, label=f"{label} train")
    axes[0].plot(epochs_range, h["val_loss"], '--', color=color, alpha=1.0, label=f"{label} val")

# Mark phase boundary
phase1_end = max(2 * EPOCHS_PER_ROUND, int(TOTAL_EPOCHS * 0.20))
axes[0].axvline(x=phase1_end + 0.5, color='gray', linestyle=':', alpha=0.5, label='Phase 1/2 boundary')
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title(f'Training & Validation Loss (Dirichlet alpha={ALPHA})', fontsize=13, fontweight='bold')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(True, alpha=0.3)

# --- Accuracy Plot ---
for model_type in MODEL_PARTITION_MAP.keys():
    if model_type not in training_history:
        continue
    h = training_history[model_type]
    color = colors.get(model_type, 'gray')
    partition_id = MODEL_PARTITION_MAP[model_type]
    label = f"{model_type} (P{partition_id})"
    axes[1].plot(epochs_range, [a * 100 for a in h["train_acc"]], '-', color=color, alpha=0.6, label=f"{label} train")
    axes[1].plot(epochs_range, [a * 100 for a in h["val_acc"]], '--', color=color, alpha=1.0, label=f"{label} val")

axes[1].axvline(x=phase1_end + 0.5, color='gray', linestyle=':', alpha=0.5, label='Phase 1/2 boundary')
axes[1].axhline(y=50, color='red', linestyle=':', alpha=0.3, label='Random baseline (50%)')
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_title(f'Training & Validation Accuracy (Dirichlet alpha={ALPHA})', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([40, 105])

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, f"training_curves_alpha_{ALPHA}.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {CHECKPOINT_DIR}/training_curves_alpha_{ALPHA}.png")